In [3]:
from resources.helper.helper_dataset import get_dataloaders_cifar10
from resources.helper.helper_evaluation import set_all_seeds, set_deterministic
from resources.helper.helper_train import train_model
from resources.helper.helper_plotting import plot_training_loss, plot_accuracy, show_examples

import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision

In [4]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [5]:
set_all_seeds(42)

In [6]:
resize_transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize((70, 70)),
        torchvision.transforms.RandomCrop((64, 64)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))])
test_resize_transform = torchvision.transforms.Compose(
    [
        torchvision.transforms.Resize((70, 70)),
        torchvision.transforms.CenterCrop((64, 64)),
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))])

train_loader, valid_loader, test_loader = get_dataloaders_cifar10(batch_size=256,validation_fraction=0.1,
                                                                train_transforms=resize_transform,test_transforms=resize_transform)

100%|██████████| 170M/170M [00:17<00:00, 9.55MB/s]


In [7]:
class Vgg16(torch.nn.Module):
    def __init__(self,classes):
        super().__init__()

        self.block_1 = torch.nn.Sequential(
                torch.nn.Conv2d(in_channels=3,
                                out_channels=64,
                                kernel_size=(3, 3),
                                stride=(1, 1),
                                padding=1),
                torch.nn.ReLU(),
                torch.nn.Conv2d(in_channels=64,
                                out_channels=64,
                                kernel_size=(3, 3),
                                stride=(1, 1),
                                padding=1),
                torch.nn.ReLU(),
                torch.nn.MaxPool2d(kernel_size=(2, 2),
                                   stride=(2, 2))
        )

        self.block_2 = torch.nn.Sequential(
                torch.nn.Conv2d(in_channels=64,
                                out_channels=128,
                                kernel_size=(3, 3),
                                stride=(1, 1),
                                padding=1),
                torch.nn.ReLU(),
                torch.nn.Conv2d(in_channels=128,
                                out_channels=128,
                                kernel_size=(3, 3),
                                stride=(1, 1),
                                padding=1),
                torch.nn.ReLU(),
                torch.nn.MaxPool2d(kernel_size=(2, 2),
                                   stride=(2, 2))
        )

        self.block_3 = torch.nn.Sequential(
                torch.nn.Conv2d(in_channels=128,
                                out_channels=256,
                                kernel_size=(3, 3),
                                stride=(1, 1),
                                padding=1),
                torch.nn.ReLU(),
                torch.nn.Conv2d(in_channels=256,
                                out_channels=256,
                                kernel_size=(3, 3),
                                stride=(1, 1),
                                padding=1),
                torch.nn.ReLU(),
                torch.nn.Conv2d(in_channels=256,
                                out_channels=256,
                                kernel_size=(3, 3),
                                stride=(1, 1),
                                padding=1),
                torch.nn.ReLU(),
                torch.nn.MaxPool2d(kernel_size=(2, 2),
                                   stride=(2, 2))
        )


        self.block_4 = torch.nn.Sequential(
                torch.nn.Conv2d(in_channels=256,
                                out_channels=512,
                                kernel_size=(3, 3),
                                stride=(1, 1),
                                padding=1),
                torch.nn.ReLU(),
                torch.nn.Conv2d(in_channels=512,
                                out_channels=512,
                                kernel_size=(3, 3),
                                stride=(1, 1),
                                padding=1),
                torch.nn.ReLU(),
                torch.nn.Conv2d(in_channels=512,
                                out_channels=512,
                                kernel_size=(3, 3),
                                stride=(1, 1),
                                padding=1),
                torch.nn.ReLU(),
                torch.nn.MaxPool2d(kernel_size=(2, 2),
                                   stride=(2, 2))
        )

        self.block_5 = torch.nn.Sequential(
                torch.nn.Conv2d(in_channels=512,
                                out_channels=512,
                                kernel_size=(3, 3),
                                stride=(1, 1),
                                padding=1),
                torch.nn.ReLU(),
                torch.nn.Conv2d(in_channels=512,
                                out_channels=512,
                                kernel_size=(3, 3),
                                stride=(1, 1),
                                padding=1),
                torch.nn.ReLU(),
                torch.nn.Conv2d(in_channels=512,
                                out_channels=512,
                                kernel_size=(3, 3),
                                stride=(1, 1),
                                padding=1),
                torch.nn.ReLU(),
                torch.nn.MaxPool2d(kernel_size=(2, 2),
                                   stride=(2, 2))
        )

        height, width = 3, 3
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(512*height*width, 4096),
            torch.nn.ReLU(True),
            torch.nn.Dropout(p=0.5),
            torch.nn.Linear(4096, 4096),
            torch.nn.ReLU(True),
            torch.nn.Dropout(p=0.5),
            torch.nn.Linear(4096, classes),
        )

        for m in self.modules():
            if isinstance(m, torch.torch.nn.Conv2d) or isinstance(m, torch.torch.nn.Linear):
                torch.nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    m.bias.detach().zero_()

        self.avgpool = torch.nn.AdaptiveAvgPool2d((height, width))


    def forward(self, x):
        x=self.block_1(x)
        x=self.block_2(x)
        x=self.block_3(x)
        x=self.block_4(x)
        x=self.block_5(x)
        x=self.avgpool(x)
        x=x.view(x.size(0),-1)
        logits=self.classifier(x)
        return logits






In [8]:
model=Vgg16(10)
model=model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,factor=0.1, mode='max')
Epochs=50
minibatch_loss_list, train_acc_list, valid_acc_list = train_model(
    model=model,
    num_epochs=Epochs,
    train_loader=train_loader,
    valid_loader=valid_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    device=device,
    logging_interval=50,
    scheduler=scheduler,
    scheduler_on='valid_acc')

plot_training_loss(minibatch_loss_list=minibatch_loss_list,
                   num_epochs=Epochs,
                   iter_per_epoch=len(train_loader),
                   results_dir=None,
                   averaging_iterations=20)

plt.show()

plot_accuracy(train_acc_list=train_acc_list,
              valid_acc_list=valid_acc_list,
              results_dir=None)

plt.ylim([80, 100])
plt.show()

Epoch: 001/050 | Batch 0000/0175 | Loss: 3.7228
Epoch: 001/050 | Batch 0050/0175 | Loss: 2.2868
Epoch: 001/050 | Batch 0100/0175 | Loss: 2.2020
Epoch: 001/050 | Batch 0150/0175 | Loss: 2.1694
Epoch: 001/050 | Train: 10.05% | Validation: 9.58%
Time elapsed: 1.60 min
Last lr:  [0.1]
Epoch: 002/050 | Batch 0000/0175 | Loss: 2.3029
Epoch: 002/050 | Batch 0050/0175 | Loss: 2.2635
Epoch: 002/050 | Batch 0100/0175 | Loss: 2.1970
Epoch: 002/050 | Batch 0150/0175 | Loss: 2.0519
Epoch: 002/050 | Train: 24.31% | Validation: 23.78%
Time elapsed: 3.19 min
Last lr:  [0.1]
Epoch: 003/050 | Batch 0000/0175 | Loss: 2.0746
Epoch: 003/050 | Batch 0050/0175 | Loss: 2.1377
Epoch: 003/050 | Batch 0100/0175 | Loss: 2.2984
Epoch: 003/050 | Batch 0150/0175 | Loss: 1.6996
Epoch: 003/050 | Train: 30.97% | Validation: 31.62%
Time elapsed: 4.78 min
Last lr:  [0.1]
Epoch: 004/050 | Batch 0000/0175 | Loss: 1.9515
Epoch: 004/050 | Batch 0050/0175 | Loss: 1.7960
Epoch: 004/050 | Batch 0100/0175 | Loss: 1.9745
Epoch: 0

KeyboardInterrupt: 

In [10]:
from google.colab import drive
drive.mount('/content/drive')

import os
save_path = '/content/drive/My Drive/models/vgg16/'
os.makedirs(save_path, exist_ok=True)

torch.save(model.state_dict(), save_path + 'vgg16.pt')
torch.save(optimizer.state_dict(), save_path + 'optimizer.pt')
torch.save(scheduler.state_dict(), save_path + 'scheduler.pt')

Mounted at /content/drive


In [ ]:
import json
import os

# Assuming these lists already exist
training_stats = {
    "minibatch_loss_list": minibatch_loss_list,
    "train_acc_list": train_acc_list,
    "valid_acc_list": valid_acc_list
}

save_path = '/content/drive/My Drive/models/lenet_5/'
os.makedirs(save_path, exist_ok=True)

with open(save_path + 'training_stats.json', 'w') as f:
    json.dump(training_stats, f, indent=4)
